In [54]:
import pandas as pd
import numpy as np

In [55]:
df = pd.read_csv("animal_data_dirty1.csv")
df.head()

,Animal type,Country,Weight kg,Body Length cm,Gender,Animal code,Latitude,Longitude,Animal name,Observation date
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,03.01.2024
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,03.02.2024
2,European bison,Poland,930.0,335.0,male,NaN,52.828845,23.820144,Szefu,01.03.2024
3,European bison,Poland,909.0,311.0,not determined,NaN,52.830509,23.826849,NaN,01.03.2024
4,European bison™,Poland,581.0,277.0,female,NaN,52.834109,23.807093,NaN,01.03.2024


In [56]:
print("Filas y Columnas", df.shape)

Filas y Columnas (1011, 10)


In [57]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1011 entries, 0 to 1010
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Animal type       991 non-null    str    
 1   Country           999 non-null    str    
 2   Weight kg         984 non-null    float64
 3   Body Length cm    984 non-null    float64
 4   Gender            992 non-null    str    
 5   Animal code       0 non-null      float64
 6   Latitude          913 non-null    float64
 7   Longitude         913 non-null    float64
 8   Animal name       52 non-null     str    
 9   Observation date  1011 non-null   str    
dtypes: float64(5), str(5)
memory usage: 79.1 KB


In [58]:
df.isnull().sum()

Animal type           20
Country               12
Weight kg             27
Body Length cm        27
Gender                19
Animal code         1011
Latitude              98
Longitude             98
Animal name          959
Observation date       0
dtype: int64

In [59]:
df.duplicated().sum()

np.int64(167)

In [60]:
#Limpiar

In [61]:
df["Animal type"].value_counts(dropna=False)

df["Gender"].value_counts(dropna=False)

df["Country"].value_counts(dropna=False)

Country
Poland            291
Germany           176
Slovakia          151
Hungary           146
Czech Republic    103
Austria            74
PL                 15
NaN                12
HU                 11
Hungry             10
CZ                  5
DE                  5
Czech               4
Australia           4
CC                  4
Name: count, dtype: int64

In [62]:
df = df.drop_duplicates()
print("Filas y Columnas duplicadas", (df.duplicated().sum()))

Filas y Columnas duplicadas 0


In [63]:
df_clean = df.copy()
df_clean = df_clean.drop_duplicates()
df_clean = df_clean.dropna(subset=["Animal type"])
df_clean = df_clean.drop(columns=["Animal code"])

In [64]:
#Se descarta 'Animal code' por presentar un alto porcentaje de valores nulos
# y no constituir una variable necesaria para los análisis morfométricos y geográficos.

In [65]:
df_clean["Animal type"].value_counts(dropna=False)

Animal type
red squirrel       387
hedgehog           274
lynx                61
European bison      49
red squirrell       16
lynx?               10
red squirel         10
European bison™      6
European bisson      4
European buster      4
ledgehod             3
wedgehod             1
Name: count, dtype: int64

In [66]:
#Hay muchos errores detectados en la columna entonces decidí crear un diccionario para mapear los 
#valores corrector y reemplazarlos.

In [67]:
animal_mapping = {
    "red squirell": "red squirrel",
    "red squirel": "red squirrel",
    "red squirrell": "red squirrel",


    "ledgehod": "hedgehog",
    "wedgehod": "hedgehog",

    "lynx?": "lynx",

    "European bison™": "European bison",
    "European bisson": "European bison",
    "European buster": "European bison",
    "eurpean bison": "European bison",
}

In [68]:
df_clean["Animal type"] = (
    df_clean["Animal type"].replace(animal_mapping)
    .str.strip()
    .str.lower()
)

In [69]:
df_clean["Animal type"].value_counts() 

Animal type
red squirrel      413
hedgehog          278
lynx               71
european bison     63
Name: count, dtype: int64

In [70]:
df_clean["Country"].value_counts(dropna=False)

Country
Poland            182
Germany           173
Slovakia          151
Hungary           103
Czech Republic    103
Austria            66
PL                 10
HU                  8
Hungry              7
CZ                  5
DE                  5
Czech               4
Australia           4
CC                  4
Name: count, dtype: int64

In [71]:
#Lo mismo pasa con la columna de países, tal vez esta es una manera más larga pero me aseguro de
#que esté mapeado correctamente y cuando los biologos quieran hacer un análisis de los animales 
#no tengan problemas con la lógica de los datos.

In [72]:
df_clean[df_clean["Country"] == "CC"]

,Animal type,Country,Weight kg,Body Length cm,Gender,Latitude,Longitude,Animal name,Observation date
397,european bison,CC,603.0,291.0,male,50.644813,14.839817,NaN,22.03.2024
398,european bison,CC,717.0,305.0,male,50.635565,14.839320,NaN,22.03.2024
661,european bison,CC,560.0,250.0,female,50.645077,14.840304,NaN,08.04.2024
777,european bison,CC,656.0,245.0,female,50.643056,14.843037,NaN,19.04.2024


In [73]:
df_clean["Country"] = df_clean["Country"].astype(str).str.strip()

In [74]:
country_mapping = {
    "PL": "Poland",

    "DE": "Germany",

    "HU": "Hungary",
    "Hungry": "Hungary",

    "CZ": "Czech Republic",
    "Czech": "Czech Republic",
    "CC": "Czech Republic",
    
    "Australia": "Austria"
}

In [75]:
df_clean["Country"] = df_clean["Country"].replace(country_mapping)

In [76]:
df_clean["Country"].value_counts(dropna=False)

Country
Poland            192
Germany           178
Slovakia          151
Hungary           118
Czech Republic    116
Austria            70
Name: count, dtype: int64

In [77]:
df_clean.groupby("Animal type")[["Weight kg", "Body Length cm"]].describe().T

Animal type           european bison    hedgehog        lynx  red squirrel
Weight kg      count       63.000000  275.000000   71.000000    407.000000
               mean       588.396825    0.797455   23.267606      0.293523
               std        244.028723    0.225623   18.401830      0.054714
               min         24.000000    0.400000   11.000000     -0.252000
               25%        437.000000    0.600000   18.000000      0.275000
               50%        603.000000    0.800000   22.000000      0.298000
               75%        722.000000    1.000000   25.000000      0.320000
               max       1100.000000    1.200000  171.000000      0.360000
Body Length cm count       63.000000  275.000000   71.000000    407.000000
               mean       244.238095   19.952727   82.901408     20.329238
               std         62.371084    9.301987   15.228878      3.316581
               min        100.000000  -19.000000   52.000000    -19.000000
               25%        211.000000   17.000000   73.000000     19.000000
               50%        255.000000   20.000000   82.000000     20.000000
               75%        289.500000   22.000000   91.000000     22.000000
               max        350.000000  151.000000  131.000000     24.000000

In [78]:
#A mí estos datos me parecen bastante alterados, tienen errores tipográficos (signos negativos, comas, puntos etc) y también
#outliers que no hacen sentido, por ejemplo un animal de una especie grande con pesos de 24 kg o especies pequeñas con pesos de 900kg.
#Eso me da a pensar que hay que revisarlo por especie y de esa manera corregir un dato que se digitó pensando en gramos en la columna de kg.

In [79]:
df_clean[df_clean["Weight kg"] < 0]

,Animal type,Country,Weight kg,Body Length cm,Gender,Latitude,Longitude,Animal name,Observation date
457,red squirrel,Hungary,-0.251,20.0,female,47.508618,18.944915,NaN,26.03.2024
503,red squirrel,Hungary,-0.250,19.0,female,47.509538,18.945968,NaN,31.03.2024
663,red squirrel,Slovakia,-0.252,21.0,male,49.057599,22.422203,NaN,08.04.2024


In [80]:
df_clean[df_clean["Weight kg"] < 0]

,Animal type,Country,Weight kg,Body Length cm,Gender,Latitude,Longitude,Animal name,Observation date
457,red squirrel,Hungary,-0.251,20.0,female,47.508618,18.944915,NaN,26.03.2024
503,red squirrel,Hungary,-0.250,19.0,female,47.509538,18.945968,NaN,31.03.2024
663,red squirrel,Slovakia,-0.252,21.0,male,49.057599,22.422203,NaN,08.04.2024


In [81]:
df_clean["Weight kg"] = df_clean["Weight kg"].abs()
df_clean["Body Length cm"] = df_clean["Body Length cm"].abs()

In [82]:
df_clean[df_clean["Weight kg"] < 0]

,Animal type,Country,Weight kg,Body Length cm,Gender,Latitude,Longitude,Animal name,Observation date


In [83]:
df_clean[df_clean["Weight kg"] < 0]

,Animal type,Country,Weight kg,Body Length cm,Gender,Latitude,Longitude,Animal name,Observation date


In [84]:
#Con esto dejo corregidos valores negativos que estaban afectando los posibles resultados de análisis de la base.

In [85]:
df_clean[(df_clean["Animal type"] == "hedgehog") & (df_clean["Body Length cm"] > 40)]

,Animal type,Country,Weight kg,Body Length cm,Gender,Latitude,Longitude,Animal name,Observation date
430,hedgehog,Germany,0.9,151.0,female,52.583004,13.150686,NaN,24.03.2024


In [86]:
df_clean[(df_clean["Animal type"] == "european bison") & (df_clean["Weight kg"] < 100)]

,Animal type,Country,Weight kg,Body Length cm,Gender,Latitude,Longitude,Animal name,Observation date
208,european bison,Czech Republic,30.0,100.0,female,50.636423,14.847287,NaN,07.03.2024
248,european bison,Poland,24.0,100.0,female,52.834112,23.825722,NaN,11.03.2024
312,european bison,Poland,35.0,125.0,female,52.820739,23.815145,NaN,16.03.2024
379,european bison,Czech Republic,35.0,110.0,male,50.640733,14.845160,NaN,21.03.2024


In [87]:
df_clean[(df_clean["Animal type"] == "lynx") & (df_clean["Weight kg"] > 40)]

,Animal type,Country,Weight kg,Body Length cm,Gender,Latitude,Longitude,Animal name,Observation date
364,lynx,Poland,171.0,75.0,male,52.831229,23.814123,NaN,21.03.2024


In [88]:
df_clean.loc[430, "Body Length cm"] = np.nan
df_clean.loc[364, "Weight kg"] = np.nan

df_clean["Weight kg"] = df_clean.groupby("Animal type")["Weight kg"].transform(lambda x: x.fillna(x.mean()))
df_clean["Body Length cm"] = df_clean.groupby("Animal type")["Body Length cm"].transform(lambda x: x.fillna(x.mean()))

print("Nulos en Weight kg:", df_clean["Weight kg"].isnull().sum())
print("Nulos en Body Length cm:", df_clean["Body Length cm"].isnull().sum())

Nulos en Weight kg: 0
Nulos en Body Length cm: 0


In [89]:
df_clean.groupby("Animal type")[["Weight kg", "Body Length cm"]].describe().T

Animal type           european bison    hedgehog        lynx  red squirrel
Weight kg      count       63.000000  278.000000   71.000000    413.000000
               mean       588.396825    0.797455   21.157143      0.297224
               std        244.028723    0.224398    4.731764      0.028118
               min         24.000000    0.400000   11.000000      0.200000
               25%        437.000000    0.600000   18.000000      0.275000
               50%        603.000000    0.800000   22.000000      0.298000
               75%        722.000000    1.000000   24.500000      0.320000
               max       1100.000000    1.200000   30.000000      0.360000
Body Length cm count       63.000000  278.000000   71.000000    413.000000
               mean       244.238095   19.729927   82.901408     20.515971
               std         62.371084    3.668588   15.228878      1.817957
               min        100.000000   11.000000   52.000000     14.000000
               25%        211.000000   17.000000   73.000000     19.000000
               50%        255.000000   20.000000   82.000000     20.515971
               75%        289.500000   22.000000   91.000000     22.000000
               max        350.000000   32.000000  131.000000     24.000000

In [90]:
#Hacer una variable. Como tenemos 4 tipos de animales y a mi opinion 2 son grandes y 2 especies pequeñas puedo clasificarlos así.

In [91]:
df_clean["size_category"] = np.where(
    df_clean["Animal type"].isin(["european bison", "lynx"]),
    "Large",
    "Small"
)

In [92]:
df_clean["size_category"].value_counts()

size_category
Small    691
Large    134
Name: count, dtype: int64

In [93]:
#Lo que quiere decir que de todos los animales en avistados en Europa central y Oriental la gran mayoría es una especie considerada pequeña.

In [94]:
#Para la función de numpy se puede buscar la longitud con mean y la desviación estándar

In [95]:
longitud_lince = df_clean[df_clean["Animal type"] == "lynx"]["Body Length cm"]
media_lince = np.mean(longitud_lince)
std_lince = np.std(longitud_lince)

print(f"Promedio de longitud del Lince: {media_lince:.2f} cm")
print(f"Desviación estándar: {std_lince:.2f} cm")

Promedio de longitud del Lince: 82.90 cm
Desviación estándar: 15.12 cm


In [96]:
#La longitud promedio del lince es de 82.90 cm y tiene una desviación estándar de 15,12 cm. Eso lo que nos deja ver es que de los avistados
#En Europa central y Oriental durante la recolección de datos la mayoría de los ejemplares se concentran entre los 67 y 98 cm de longitud.

In [97]:
#Numpy 

In [98]:
#1: Número de observaciones por tipo de animal
df_clean.groupby("Animal type")["Animal type"].count().rename("Total observaciones")

Animal type
european bison     63
hedgehog          278
lynx               71
red squirrel      413
Name: Total observaciones, dtype: int64

In [99]:
#Esto nos permite afirmar que el animal más observado durante los periodos fue la ardilla roja con 413 avistamientos, 
#luego el Erizo con 278 avistamientos, seguido de el Lince y por último el bisonte. Podría existir una relación con los tamaños pues también
#los más avistados fueron los de menor tamaño y se ve como una variable inversamente proporcional.

In [100]:
# 2. Promedio de peso según el animal
df_clean.groupby("Animal type")["Weight kg"].mean().round(2).rename("Peso promedio (kg)")

Animal type
european bison    588.40
hedgehog            0.80
lynx               21.16
red squirrel        0.30
Name: Peso promedio (kg), dtype: float64

In [101]:
#Es una variable relacionada directamente a su tamaño y tiene diferencias muy evidentes entre especies: el bisonte lidera ampliamente con 588.40kg
#seguido del lince con 21.16kg, el erizo con 0.80kg y opr último 0.30 kg de la ardilla. Esta diferencia tan grande nos confirma que los datos
#quedaron en efecto limpios y biológicamente posibles de usar en un estudio más amplio. Y confirma mi teoría de que es mejor tratar los datos
#de manera segmentada pues el todo no es limpiarlos sino que sean útiles para la empresa o quien me contrate como analista.

In [102]:
# 3. Peso promedio desglosado por país y especie
df_clean.groupby(["Country", "Animal type"])["Weight kg"].mean().round(2).unstack()

Animal type,european bison,hedgehog,lynx,red squirrel
Country,,,,
Austria,NaN,0.78,22.20,0.30
Czech Republic,526.06,0.78,21.25,0.30
Germany,NaN,0.77,NaN,0.29
Hungary,NaN,0.84,21.46,0.29
Poland,641.53,0.81,19.96,0.31
Slovakia,474.89,0.80,21.65,0.30


In [103]:
#Los valoes NaN de la tabla no corresponden a datos faltantes del dataset, sino a la ausencia de avistamientos de ciertas especies en 
# determinados países, por ejemplo, bisones en Austria, Alemania o Hungría o linces en Alemania. En los países en los que sí se encuentran 
# más de un ejemplar, el peso promedio por especie se mantiene dentro de los parámetros normales.

In [104]:
print("Filas y Columnas", df.shape)

Filas y Columnas (844, 10)


In [105]:
#4. Promedio de peso por el género del animal
df_clean.groupby(["Animal type", "Gender"])["Weight kg"].mean().round(2).unstack()

Gender,female,male,not determined
Animal type,,,
european bison,481.11,728.00,651.00
hedgehog,0.75,0.84,0.95
lynx,21.00,21.33,NaN
red squirrel,0.30,0.30,0.30


In [ ]:
#Con esta variable podemos observar una variación grande dependiente del género del animal en especies
# como el bisonte donde la hembra promedia un 481.11 kg mientras que el macho 728 kg. En especies como
# el lince y el erizo, esta variación aunque sigue siendo a favor del macho no es tan amplia. Por
# su parte las ardillas tienen un promedio de peso igual, independientemente del género del animal.
# Respecto a la columna no determinada es comprensible pues son avistamientos de lejos y no en todos se
# puede reconocer el género de la especie, o como en el caso del lince "NaN" no hay registro no 
#determinado.